# Phase 3C: Fine-Tuning, Calibration & Leakage Mitigation

**Project**: Neural PK-PD Modeling with Physics-Informed Neural ODEs  
**Date**: February 24, 2026  
**Previous Phase**: [phase3b_model_design.ipynb](phase3b_model_design.ipynb)  
**Next Phase**: [phase3d_experiments.ipynb](phase3d_experiments.ipynb)  

---

## 🎯 Phase 3C Objectives

1. **Task-Specific Fine-Tuning** — Freeze encoder, tune hERG/Caco-2 heads
2. **Threshold Calibration** — Optimize decision thresholds for production
3. **Split-Leakage Mitigation** — Deduplicate and retrain on clean splits
4. **Production Report** — Locked-threshold metrics for all tasks


---
## 1. Environment Setup & Load Phase 3B Artifacts


In [1]:
# ============================================
# IMPORTS & LOAD PHASE 3B ARTIFACTS
# ============================================

# Core Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Neural ODE
from torchdiffeq import odeint

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    roc_auc_score, accuracy_score, f1_score, classification_report, roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import pickle, os, json as _json, datetime, platform, sys, hashlib

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')

# ── Execution Timestamp Logger ──────────────────────────────────
CELL_EXEC_LOG: list = []

def log_cell_start(section: str) -> None:
    now = datetime.datetime.now()
    CELL_EXEC_LOG.append({'section': section, 'start': now, 'end': None, 'duration_s': None})
    print(f'\u23f1  [{now.strftime("%Y-%m-%d %H:%M:%S")}] Starting: {section}')

def log_cell_end() -> None:
    now = datetime.datetime.now()
    if CELL_EXEC_LOG and CELL_EXEC_LOG[-1]['end'] is None:
        entry = CELL_EXEC_LOG[-1]
        entry['end'] = now
        entry['duration_s'] = round((now - entry['start']).total_seconds(), 2)
        print(f'\u23f1  [{now.strftime("%Y-%m-%d %H:%M:%S")}] Completed in {entry["duration_s"]:.1f}s')

def log_elapsed(label: str) -> None:
    elapsed = (datetime.datetime.now() - SESSION_START).total_seconds()
    h, rem = divmod(int(elapsed), 3600)
    m, s = divmod(rem, 60)
    print(f'\u23f1  [{label}] Elapsed: {h}h {m}m {s}s')

SESSION_START = datetime.datetime.now()

# ── Load Phase 3B Artifacts ────────────────────────────────────
log_cell_start('Load Phase 3B Artifacts')

ARTIFACT_DIR_3A = 'data/processed/phase3a_artifacts'
ARTIFACT_DIR_3B = 'data/processed/phase3b_artifacts'

# Config
with open(os.path.join(ARTIFACT_DIR_3B, 'config.json'), 'r') as f:
    config = _json.load(f)

# Feature metadata
with open(os.path.join(ARTIFACT_DIR_3B, 'feature_meta.pkl'), 'rb') as f:
    feature_meta = pickle.load(f)
    FEATURE_COLS = feature_meta['FEATURE_COLS']
    PHYSICO_FEATURES = feature_meta['PHYSICO_FEATURES']
    FINGERPRINT_COLS = feature_meta['FINGERPRINT_COLS']
    N_BITS = feature_meta['N_BITS']
    FEAT_DIM = feature_meta['FEAT_DIM']

# Loader state
with open(os.path.join(ARTIFACT_DIR_3B, 'loader_state.pkl'), 'rb') as f:
    loader_state = pickle.load(f)
    tasks_raw = loader_state['tasks_raw']
    task_types = loader_state['task_types']
    feat_scalers = loader_state['feat_scalers']
    target_scalers = loader_state['target_scalers']

# Model info
with open(os.path.join(ARTIFACT_DIR_3B, 'model_info.pkl'), 'rb') as f:
    model_info = pickle.load(f)
    total_params = model_info['total_params']
    trainable_params = model_info['trainable_params']

# Raw arrays
with open(os.path.join(ARTIFACT_DIR_3B, 'raw_arrays.pkl'), 'rb') as f:
    raw_arrays = pickle.load(f)
    X_binding = raw_arrays['X_binding']; y_binding = raw_arrays['y_binding']
    X_herg = raw_arrays['X_herg']; y_herg = raw_arrays['y_herg']
    X_caco2 = raw_arrays['X_caco2']; y_caco2 = raw_arrays['y_caco2']
    X_clearance = raw_arrays['X_clearance']; y_clearance = raw_arrays['y_clearance']

# Phase 2 features
phase2_features = pd.read_parquet(os.path.join(ARTIFACT_DIR_3A, 'phase2_features.parquet'))
PHASE2_FEATURE_PATH = 'data/processed/phase2_multitask_features_with_fingerprints.csv'

# Training history
with open(os.path.join(ARTIFACT_DIR_3B, 'history.pkl'), 'rb') as f:
    history = pickle.load(f)

print(f'\n\u2705 Phase 3B artifacts loaded')
print(f'   Model params: {total_params:,}')
print(f'   Tasks: {list(tasks_raw.keys())}')
print(f'   Feature dim: {FEAT_DIM}')

log_cell_end()


PyTorch version: 2.10.0
Device: cpu
⏱  [2026-03-22 17:53:59] Starting: Load Phase 3B Artifacts

✅ Phase 3B artifacts loaded
   Model params: 302,694
   Tasks: ['binding', 'herg', 'caco2', 'clearance']
   Feature dim: 2051
⏱  [2026-03-22 17:53:59] Completed in 0.3s


---
## 2. Rebuild Model & DataLoaders


In [2]:

# ============================================
# REBUILD MODEL ARCHITECTURE & DATALOADERS
# ============================================
log_cell_start('Rebuild Model & DataLoaders')

# ── Model class definitions (MUST match phase3b exactly) ──────
class SharedEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.encoder(x)

class RegressionHead(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Linear(latent_dim // 2, 1),
        )
    def forward(self, x):
        return self.head(x)

class DeepRegressionHead(nn.Module):
    def __init__(self, latent_dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x):
        return self.head(x)

class ClassificationHead(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2),
            nn.ReLU(),
            nn.Linear(latent_dim // 2, 1),
        )
    def forward(self, x):
        return self.head(x)

class PKODEFunc(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.pk_params = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 2),  # [CL, V]
        )
        self.latent = None
    def set_latent(self, latent):
        self.latent = latent
    def forward(self, t, C):
        if self.latent is None:
            raise ValueError("Latent features not set.")
        pk = self.pk_params(self.latent)
        CL = torch.exp(pk[:, 0:1])
        V  = torch.exp(pk[:, 1:2])
        k  = CL / V
        return -k * C

class MultiTaskPKPDModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        input_dim  = config['input_dim']
        hidden_dim = config['hidden_dim']
        latent_dim = config['latent_dim']
        dropout    = config['dropout']
        self.encoder = SharedEncoder(input_dim, hidden_dim, latent_dim, dropout)
        self.binding_head   = DeepRegressionHead(latent_dim, hidden_dim=config.get('reg_head_hidden', 64), dropout=config.get('reg_head_dropout', 0.1))
        self.herg_head      = ClassificationHead(latent_dim)
        self.caco2_head     = ClassificationHead(latent_dim)
        self.clearance_head = RegressionHead(latent_dim)
        self.ode_func = PKODEFunc(latent_dim)
    def forward(self, x, task=None):
        latent = self.encoder(x)
        if task == 'binding':   return self.binding_head(latent)
        elif task == 'herg':    return self.herg_head(latent)
        elif task == 'caco2':   return self.caco2_head(latent)
        elif task == 'clearance': return self.clearance_head(latent)
        elif task == 'all':
            return {'binding': self.binding_head(latent), 'herg': self.herg_head(latent),
                    'caco2': self.caco2_head(latent), 'clearance': self.clearance_head(latent), 'latent': latent}
        else: return latent
    def predict_pk_curve(self, x, t_span, C0=1.0):
        latent = self.encoder(x)
        self.ode_func.set_latent(latent)
        C_init = torch.full((x.shape[0], 1), C0, device=x.device)
        return odeint(self.ode_func, C_init, t_span)

# ── Instantiate and load weights ──────────────────────────────
model = MultiTaskPKPDModel(config).to(device)
model.load_state_dict(torch.load(os.path.join(ARTIFACT_DIR_3B, 'model_state.pt'),
                                  map_location=device, weights_only=True))
model.eval()
print(f'Model loaded: {total_params:,} params')

# ── Rebuild DataLoaders ───────────────────────────────────────
def compute_pos_weight(y_binary):
    y_binary = np.asarray(y_binary).astype(np.float32)
    pos = float((y_binary == 1).sum())
    neg = float((y_binary == 0).sum())
    return max(1.0, neg / pos) if pos > 0 else 1.0

def make_loaders(X, y, config, is_regression=True, random_state=42):
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=config['test_size'], random_state=random_state)
    val_frac = config['val_size'] / (1 - config['test_size'])
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=val_frac, random_state=random_state)
    feat_scaler = StandardScaler()
    X_train = feat_scaler.fit_transform(X_train)
    X_val   = feat_scaler.transform(X_val)
    X_test  = feat_scaler.transform(X_test)
    target_scaler = None
    if is_regression:
        target_scaler = StandardScaler()
        y_train = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val   = target_scaler.transform(y_val.reshape(-1, 1)).flatten()
        y_test  = target_scaler.transform(y_test.reshape(-1, 1)).flatten()
    def to_loader(Xa, ya, shuffle=True):
        ds = TensorDataset(torch.FloatTensor(Xa), torch.FloatTensor(ya).unsqueeze(1))
        def collate(batch):
            Xb, yb = zip(*batch)
            return {'X': torch.stack(Xb), 'y': torch.stack(yb)}
        return DataLoader(ds, batch_size=config['batch_size'], shuffle=shuffle, collate_fn=collate)
    return (to_loader(X_train, y_train, True), to_loader(X_val, y_val, False),
            to_loader(X_test, y_test, False), feat_scaler, target_scaler)

train_loaders, val_loaders, test_loaders = {}, {}, {}
feat_scalers, target_scalers = {}, {}
for task, (X, y) in tasks_raw.items():
    tr, va, te, fs, ts = make_loaders(X, y, config, is_regression=task_types[task])
    train_loaders[task] = tr; val_loaders[task] = va; test_loaders[task] = te
    feat_scalers[task] = fs; target_scalers[task] = ts

print('DataLoaders rebuilt:')
for task, loader in train_loaders.items():
    n = len(loader.dataset)
    reg = 'regression' if task_types[task] else 'classification'
    print(f'  {task:<12}  {reg:<14}  train={n}, '
          f'val={len(val_loaders[task].dataset)}, '
          f'test={len(test_loaders[task].dataset)}')

log_cell_end()


⏱  [2026-03-22 17:54:07] Starting: Rebuild Model & DataLoaders
Model loaded: 302,694 params
DataLoaders rebuilt:
  binding       regression      train=2387, val=341, test=682
  herg          classification  train=458, val=66, test=131
  caco2         classification  train=630, val=90, test=180
  clearance     regression      train=848, val=122, test=243
⏱  [2026-03-22 17:54:07] Completed in 0.2s


---
## 3. Rebuild Core Functions

Re-define the loss, training, and validation functions needed for fine-tuning.


In [3]:

# ============================================
# REBUILD CORE FUNCTIONS (Loss + Train + Validate)
# ============================================
log_cell_start('Rebuild Core Functions')

class MultiTaskLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w = {k.replace('w_', ''): v for k, v in config.items() if k.startswith('w_')}
        self.mse = nn.MSELoss()
        self.focal_gamma = config.get('focal_gamma', 2.0)
        self.use_focal = config.get('use_focal_for_classification', True)
        hpw = config.get('herg_pos_weight', 1.0)
        cpw = config.get('caco2_pos_weight', 1.0)
        self.bce_herg  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([hpw]))
        self.bce_caco2 = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cpw]))

    def focal_loss(self, logits, targets, gamma, bce_fn):
        bce = bce_fn(logits, targets)
        if gamma == 0:
            return bce
        pt = torch.sigmoid(logits)
        pt = torch.where(targets >= 0.5, pt, 1 - pt)
        focal_weight = (1 - pt) ** gamma
        return (focal_weight * nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none')).mean()

    def forward(self, predictions, targets, task):
        if task in ('binding', 'clearance'):
            return self.w.get(task, 1.0) * self.mse(predictions, targets)
        elif task == 'herg':
            bce_fn = self.bce_herg.to(predictions.device)
            if self.use_focal:
                return self.w.get('herg', 1.0) * self.focal_loss(predictions, targets, self.focal_gamma, bce_fn)
            return self.w.get('herg', 1.0) * bce_fn(predictions, targets)
        elif task == 'caco2':
            bce_fn = self.bce_caco2.to(predictions.device)
            if self.use_focal:
                return self.w.get('caco2', 1.0) * self.focal_loss(predictions, targets, self.focal_gamma, bce_fn)
            return self.w.get('caco2', 1.0) * bce_fn(predictions, targets)
        return self.mse(predictions, targets)


def train_epoch(model, data_loaders, optimizer, criterion, device, grad_clip=1.0):
    """Train one epoch with interleaved task sampling at the MINIMUM task size."""
    model.train()
    task_losses = {task: [] for task in data_loaders.keys()}
    n_steps = min(len(loader) for loader in data_loaders.values())
    iters = {task: iter(loader) for task, loader in data_loaders.items()}

    for _step in range(n_steps):
        for task in data_loaders.keys():
            try:
                batch = next(iters[task])
            except StopIteration:
                iters[task] = iter(data_loaders[task])
                batch = next(iters[task])

            X = batch['X'].to(device)
            y = batch['y'].to(device)

            optimizer.zero_grad()
            pred = model(X, task=task)
            loss = criterion(pred, y, task)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            task_losses[task].append(loss.item())

    avg = {task: np.mean(losses) for task, losses in task_losses.items()}
    avg['total'] = sum(avg.values())
    return avg


def train_model(model, train_loaders, val_loaders, config, device):
    """Full training loop with early stopping."""
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
    criterion = MultiTaskLoss(config)
    grad_clip = config.get('grad_clip', 1.0)

    history = {'train_loss': [], 'val_loss': [], 'val_metrics': []}
    best_val_loss = float('inf')
    patience_counter = 0

    print(f"Training for up to {config['epochs']} epochs (patience={config['patience']})...")
    print("=" * 65)

    for epoch in range(config['epochs']):
        train_losses = train_epoch(model, train_loaders, optimizer, criterion, device, grad_clip=grad_clip)
        val_results  = validate(model, val_loaders, criterion, device)
        val_loss     = sum(r['loss'] for r in val_results.values())

        scheduler.step(val_loss)
        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_loss)
        history['val_metrics'].append(val_results)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['epochs']}")
            print(f"  Train Loss: {train_losses['total']:.4f}  Val Loss: {val_loss:.4f}")
            for task, metrics in val_results.items():
                if 'r2' in metrics:
                    print(f"    {task:<10}: R²={metrics['r2']:.3f}, RMSE={metrics['rmse']:.3f}")
                else:
                    print(f"    {task:<10}: AUROC={metrics['auroc']:.3f}, Acc={metrics['accuracy']:.3f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")
    model.load_state_dict(torch.load('best_model.pt', weights_only=True))
    return model, history


def validate(model, val_loaders, criterion, device):
    model.eval()
    results = {}
    with torch.no_grad():
        for task, loader in val_loaders.items():
            all_preds, all_targets = [], []
            total_loss = 0.0
            for batch in loader:
                X = batch['X'].to(device)
                y = batch['y'].to(device)
                pred = model(X, task=task)
                loss = criterion(pred, y, task)
                total_loss += loss.item() * X.size(0)
                all_preds.append(pred.cpu().numpy())
                all_targets.append(y.cpu().numpy())
            preds = np.concatenate(all_preds).flatten()
            targets = np.concatenate(all_targets).flatten()
            n = len(targets)
            avg_loss = total_loss / n if n > 0 else 0
            if task_types.get(task, True):
                rmse = float(np.sqrt(mean_squared_error(targets, preds)))
                r2 = float(r2_score(targets, preds)) if n > 1 else 0.0
                mae = float(mean_absolute_error(targets, preds))
                results[task] = {'loss': avg_loss, 'rmse': rmse, 'r2': r2, 'mae': mae}
            else:
                probs = 1.0 / (1.0 + np.exp(-preds))
                auroc = float(roc_auc_score(targets, probs)) if len(np.unique(targets)) > 1 else 0.5
                binary_preds = (probs >= 0.5).astype(int)
                acc = float(accuracy_score(targets, binary_preds))
                f1 = float(f1_score(targets, binary_preds, zero_division=0))
                results[task] = {'loss': avg_loss, 'auroc': auroc, 'accuracy': acc, 'f1': f1}
    return results


def collect_probs_targets(model_obj, loader, task, device):
    model_obj.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            X = batch['X'].to(device)
            y = batch['y'].to(device)
            logits = model_obj(X, task=task)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            all_probs.append(probs)
            all_targets.append(y.cpu().numpy().flatten())
    return np.concatenate(all_probs), np.concatenate(all_targets)


def _precision_from_preds(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0


def _split_like_training(X, y, cfg, random_state=42):
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=cfg['test_size'], random_state=random_state)
    val_frac = cfg['val_size'] / (1 - cfg['test_size'])
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=val_frac, random_state=random_state)
    return X_train, X_val, X_test, y_train, y_val, y_test


def _row_hash(X):
    Xc = np.ascontiguousarray(X)
    return Xc.view(np.dtype((np.void, Xc.dtype.itemsize * Xc.shape[1]))).ravel()


FT_TASKS = ['herg', 'caco2']

print('✅ Core functions rebuilt: MultiTaskLoss, train_epoch, train_model, validate,')
print('   collect_probs_targets, _split_like_training, _row_hash')
log_cell_end()


⏱  [2026-03-22 17:54:14] Starting: Rebuild Core Functions
✅ Core functions rebuilt: MultiTaskLoss, train_epoch, train_model, validate,
   collect_probs_targets, _split_like_training, _row_hash
⏱  [2026-03-22 17:54:14] Completed in 0.0s


---
## 11. Task-Specific Fine-Tuning (hERG/Caco-2 with Frozen Shared Encoder)

This section performs targeted fine-tuning for classification tasks only:
- freeze shared encoder + PK ODE + regression heads
- train only `herg_head` and `caco2_head`
- optimize and monitor AUROC/F1 on validation and test splits

In [4]:

# ============================================
# FINE-TUNE SETUP (freeze backbone, cls heads only)
# ============================================
log_cell_start("Cell 22 - FineTune Setup")

FT_TASKS = ['herg', 'caco2']


def freeze_for_cls_finetune(model_obj):
    for p in model_obj.parameters():
        p.requires_grad = False

    # Unfreeze only classification heads
    for p in model_obj.herg_head.parameters():
        p.requires_grad = True
    for p in model_obj.caco2_head.parameters():
        p.requires_grad = True

    trainable = [p for p in model_obj.parameters() if p.requires_grad]
    total = sum(p.numel() for p in model_obj.parameters())
    trainable_n = sum(p.numel() for p in trainable)

    print(f"Trainable parameters (fine-tune): {trainable_n:,} / {total:,}")
    print("Frozen: encoder + ODE + regression heads")
    print("Trainable: herg_head + caco2_head")
    return trainable


# Build classification-only loaders
ft_train_loaders = {k: v for k, v in train_loaders.items() if k in FT_TASKS}
ft_val_loaders = {k: v for k, v in val_loaders.items() if k in FT_TASKS}
ft_test_loaders = {k: v for k, v in test_loaders.items() if k in FT_TASKS}

print("Fine-tune task loaders:")
for k in FT_TASKS:
    print(
        f"  {k:<6} train={len(ft_train_loaders[k].dataset)}, "
        f"val={len(ft_val_loaders[k].dataset)}, test={len(ft_test_loaders[k].dataset)}"
    )

# Optional sanity check from data-quality gate
if 'quality_df' in globals() and isinstance(quality_df, pd.DataFrame):
    q_cls = quality_df[quality_df['task'].isin(FT_TASKS)][['task', 'leak_tr_val', 'leak_tr_test', 'leak_val_test']]
    print("\nLeakage summary for fine-tune tasks:")
    display(q_cls)

log_cell_end()


⏱  [2026-03-22 17:54:18] Starting: Cell 22 - FineTune Setup
Fine-tune task loaders:
  herg   train=458, val=66, test=131
  caco2  train=630, val=90, test=180
⏱  [2026-03-22 17:54:18] Completed in 0.0s


In [5]:

# ============================================
# RUN TASK-SPECIFIC FINE-TUNING
# ============================================
log_cell_start("Cell 23 - FineTune Run")

# Start from current best model state
ft_model = MultiTaskPKPDModel(config).to(device)
ft_model.load_state_dict(model.state_dict())

trainable_params = freeze_for_cls_finetune(ft_model)

ft_config = {
    'epochs': 40,
    'patience': 10,
    'learning_rate': 3e-4,
    'weight_decay': 1e-5,
    'grad_clip': 1.0,
}

optimizer = optim.Adam(trainable_params, lr=ft_config['learning_rate'], weight_decay=ft_config['weight_decay'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
criterion_ft = MultiTaskLoss(config)

best_score = -np.inf
patience_counter = 0
ft_history = {'val_mean_auroc': [], 'val_results': []}

print("\nStarting fine-tuning (hERG + Caco-2 only)...")
print("=" * 70)

for epoch in range(ft_config['epochs']):
    _ = train_epoch(
        ft_model,
        ft_train_loaders,
        optimizer,
        criterion_ft,
        device,
        grad_clip=ft_config['grad_clip']
    )

    val_results = validate(ft_model, ft_val_loaders, criterion_ft, device)
    mean_auroc = float(np.mean([val_results[t]['auroc'] for t in FT_TASKS]))
    ft_history['val_mean_auroc'].append(mean_auroc)
    ft_history['val_results'].append(val_results)

    scheduler.step(mean_auroc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>3}/{ft_config['epochs']} | mean AUROC={mean_auroc:.4f}")
        for t in FT_TASKS:
            print(
                f"  {t:<6} AUROC={val_results[t]['auroc']:.4f} "
                f"F1={val_results[t]['f1']:.4f}"
            )

    if mean_auroc > best_score:
        best_score = mean_auroc
        patience_counter = 0
        torch.save(ft_model.state_dict(), 'best_model_finetune_hc.pt')
    else:
        patience_counter += 1
        if patience_counter >= ft_config['patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Load best fine-tuned weights
ft_model.load_state_dict(torch.load('best_model_finetune_hc.pt', weights_only=True))

print("\nBest validation mean AUROC:", round(best_score, 4))

# Final test evaluation
ft_test_results = validate(ft_model, ft_test_loaders, criterion_ft, device)

print("\nTest metrics after fine-tuning:")
for t in FT_TASKS:
    print(
        f"  {t:<6} AUROC={ft_test_results[t]['auroc']:.4f} "
        f"F1={ft_test_results[t]['f1']:.4f} Acc={ft_test_results[t]['accuracy']:.4f}"
    )

ft_summary = pd.DataFrame([
    {
        'task': t,
        'auroc': ft_test_results[t]['auroc'],
        'f1': ft_test_results[t]['f1'],
        'accuracy': ft_test_results[t]['accuracy'],
    }
    for t in FT_TASKS
])

print("\nFine-tune summary table:")
display(ft_summary)

log_cell_end()


⏱  [2026-03-22 17:54:22] Starting: Cell 23 - FineTune Run
Trainable parameters (fine-tune): 4,226 / 302,694
Frozen: encoder + ODE + regression heads
Trainable: herg_head + caco2_head

Starting fine-tuning (hERG + Caco-2 only)...
Epoch   1/40 | mean AUROC=0.7585
  herg   AUROC=0.7109 F1=0.8381
  caco2  AUROC=0.8060 F1=0.7253
Epoch   5/40 | mean AUROC=0.7581
  herg   AUROC=0.7087 F1=0.8350
  caco2  AUROC=0.8075 F1=0.7097
Epoch  10/40 | mean AUROC=0.7617
  herg   AUROC=0.7174 F1=0.8350
  caco2  AUROC=0.8060 F1=0.7174
Epoch  15/40 | mean AUROC=0.7606
  herg   AUROC=0.7152 F1=0.8350
  caco2  AUROC=0.8060 F1=0.7234
Epoch  20/40 | mean AUROC=0.7609
  herg   AUROC=0.7152 F1=0.8350
  caco2  AUROC=0.8065 F1=0.7234

Early stopping at epoch 21

Best validation mean AUROC: 0.7628

Test metrics after fine-tuning:
  herg   AUROC=0.8071 F1=0.8358 Acc=0.7481
  caco2  AUROC=0.8768 F1=0.8118 Acc=0.8222

Fine-tune summary table:


,task,auroc,f1,accuracy
0,herg,0.807082,0.835821,0.748092
1,caco2,0.876775,0.811765,0.822222


⏱  [2026-03-22 17:54:23] Completed in 1.2s


### 11.1 Fine-Tune Variant 2: Unfreeze Last Encoder Block + Classification Heads

Controlled variant:
- freeze ODE + regression heads
- unfreeze `herg_head`, `caco2_head`, and only final encoder block
- lower learning rate than Variant 1
- compare test AUROC/F1 directly against Variant 1

In [6]:

# ============================================
# FINE-TUNE VARIANT 2 (last encoder block + cls heads)
# ============================================
log_cell_start("Cell 24 - FineTune Variant2")


def freeze_for_variant2(model_obj):
    # Freeze all first
    for p in model_obj.parameters():
        p.requires_grad = False

    # Unfreeze classification heads
    for p in model_obj.herg_head.parameters():
        p.requires_grad = True
    for p in model_obj.caco2_head.parameters():
        p.requires_grad = True

    # Unfreeze last encoder block only (final Linear + ReLU block)
    # SharedEncoder layout: [0..9], where 8=Linear(hidden->latent), 9=ReLU
    for idx, layer in enumerate(model_obj.encoder.encoder):
        if idx >= 8:
            for p in layer.parameters():
                p.requires_grad = True

    trainable = [p for p in model_obj.parameters() if p.requires_grad]
    total = sum(p.numel() for p in model_obj.parameters())
    trainable_n = sum(p.numel() for p in trainable)

    print(f"Trainable parameters (variant2): {trainable_n:,} / {total:,}")
    print("Frozen: ODE + regression heads + most encoder layers")
    print("Trainable: last encoder block + herg_head + caco2_head")
    return trainable


# Start from base model checkpoint, not from Variant 1 weights
ft2_model = MultiTaskPKPDModel(config).to(device)
ft2_model.load_state_dict(model.state_dict())

ft2_trainable = freeze_for_variant2(ft2_model)

ft2_config = {
    'epochs': 60,
    'patience': 12,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'grad_clip': 1.0,
}

ft2_optimizer = optim.Adam(ft2_trainable, lr=ft2_config['learning_rate'], weight_decay=ft2_config['weight_decay'])
ft2_scheduler = optim.lr_scheduler.ReduceLROnPlateau(ft2_optimizer, mode='max', factor=0.5, patience=5)
criterion_ft2 = MultiTaskLoss(config)

ft2_best_score = -np.inf
ft2_patience_counter = 0

print("\nStarting variant-2 fine-tuning...")
print("=" * 70)

for epoch in range(ft2_config['epochs']):
    _ = train_epoch(
        ft2_model,
        ft_train_loaders,
        ft2_optimizer,
        criterion_ft2,
        device,
        grad_clip=ft2_config['grad_clip']
    )

    ft2_val_results = validate(ft2_model, ft_val_loaders, criterion_ft2, device)
    ft2_mean_auroc = float(np.mean([ft2_val_results[t]['auroc'] for t in FT_TASKS]))
    ft2_scheduler.step(ft2_mean_auroc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>3}/{ft2_config['epochs']} | mean AUROC={ft2_mean_auroc:.4f}")
        for t in FT_TASKS:
            print(
                f"  {t:<6} AUROC={ft2_val_results[t]['auroc']:.4f} "
                f"F1={ft2_val_results[t]['f1']:.4f}"
            )

    if ft2_mean_auroc > ft2_best_score:
        ft2_best_score = ft2_mean_auroc
        ft2_patience_counter = 0
        torch.save(ft2_model.state_dict(), 'best_model_finetune_hc_v2.pt')
    else:
        ft2_patience_counter += 1
        if ft2_patience_counter >= ft2_config['patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Evaluate best variant-2
ft2_model.load_state_dict(torch.load('best_model_finetune_hc_v2.pt', weights_only=True))
ft2_test_results = validate(ft2_model, ft_test_loaders, criterion_ft2, device)

print("\nVariant-2 best validation mean AUROC:", round(ft2_best_score, 4))
print("\nVariant-2 test metrics:")
for t in FT_TASKS:
    print(
        f"  {t:<6} AUROC={ft2_test_results[t]['auroc']:.4f} "
        f"F1={ft2_test_results[t]['f1']:.4f} Acc={ft2_test_results[t]['accuracy']:.4f}"
    )

# Side-by-side comparison with Variant 1 (if available)
rows = []
for t in FT_TASKS:
    row = {
        'task': t,
        'v1_auroc': ft_test_results[t]['auroc'] if 'ft_test_results' in globals() else np.nan,
        'v2_auroc': ft2_test_results[t]['auroc'],
        'delta_auroc_v2_minus_v1': (
            ft2_test_results[t]['auroc'] - ft_test_results[t]['auroc']
            if 'ft_test_results' in globals() else np.nan
        ),
        'v1_f1': ft_test_results[t]['f1'] if 'ft_test_results' in globals() else np.nan,
        'v2_f1': ft2_test_results[t]['f1'],
        'delta_f1_v2_minus_v1': (
            ft2_test_results[t]['f1'] - ft_test_results[t]['f1']
            if 'ft_test_results' in globals() else np.nan
        ),
    }
    rows.append(row)

ft_compare = pd.DataFrame(rows)
print("\nVariant comparison (test):")
display(ft_compare)

log_cell_end()


⏱  [2026-03-22 17:54:35] Starting: Cell 24 - FineTune Variant2
Trainable parameters (variant2): 12,482 / 302,694
Frozen: ODE + regression heads + most encoder layers
Trainable: last encoder block + herg_head + caco2_head

Starting variant-2 fine-tuning...
Epoch   1/60 | mean AUROC=0.7577
  herg   AUROC=0.7098 F1=0.8381
  caco2  AUROC=0.8055 F1=0.7333
Epoch   5/60 | mean AUROC=0.7618
  herg   AUROC=0.7185 F1=0.8381
  caco2  AUROC=0.8050 F1=0.7097
Epoch  10/60 | mean AUROC=0.7648
  herg   AUROC=0.7250 F1=0.8350
  caco2  AUROC=0.8046 F1=0.7097
Epoch  15/60 | mean AUROC=0.7753
  herg   AUROC=0.7435 F1=0.8235
  caco2  AUROC=0.8070 F1=0.7234
Epoch  20/60 | mean AUROC=0.7794
  herg   AUROC=0.7522 F1=0.8235
  caco2  AUROC=0.8065 F1=0.7234
Epoch  25/60 | mean AUROC=0.7817
  herg   AUROC=0.7554 F1=0.8235
  caco2  AUROC=0.8080 F1=0.7174
Epoch  30/60 | mean AUROC=0.7842
  herg   AUROC=0.7620 F1=0.8200
  caco2  AUROC=0.8065 F1=0.7174
Epoch  35/60 | mean AUROC=0.7822
  herg   AUROC=0.7598 F1=0.8200


,task,v1_auroc,v2_auroc,delta_auroc_v2_minus_v1,v1_f1,v2_f1,delta_f1_v2_minus_v1
0,herg,0.807082,0.795719,-0.011364,0.835821,0.839378,0.003557
1,caco2,0.876775,0.871589,-0.005186,0.811765,0.802326,-0.009439


⏱  [2026-03-22 17:54:36] Completed in 1.1s


### 11.2 Threshold Calibration Only (No Weight Updates)

This step does **not** change model weights.
It sweeps decision thresholds on validation predictions, locks the best threshold per task, and applies those thresholds to test predictions.

In [7]:

# ============================================
# THRESHOLD CALIBRATION SWEEP (no weight updates)
# ============================================
log_cell_start("Cell 25 - Threshold Calibration")


def collect_probs_targets(model_obj, loader, task, device):
    model_obj.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for batch in loader:
            Xb = batch['X'].to(device)
            yb = batch['y'].cpu().numpy().flatten()
            logits = model_obj(Xb, task=task)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            probs_all.append(probs)
            y_all.append(yb)
    return np.concatenate(probs_all), np.concatenate(y_all)


def sweep_threshold_f1(probs, y_true, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)
    best_thr, best_f1 = 0.5, -1.0
    for thr in grid:
        pred = (probs >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
    return best_thr, best_f1


# Choose best available fine-tuned model by validation mean AUROC
if 'ft2_best_score' in globals() and 'best_score' in globals() and ft2_best_score >= best_score:
    calib_model = ft2_model
    calib_source = 'Variant2'
elif 'best_score' in globals():
    calib_model = ft_model
    calib_source = 'Variant1'
else:
    calib_model = model
    calib_source = 'BaseModel'

print(f"Calibration source model: {calib_source}")

threshold_rows = []
calibrated_thresholds = {}

for task in FT_TASKS:
    val_probs, val_y = collect_probs_targets(calib_model, ft_val_loaders[task], task, device)
    test_probs, test_y = collect_probs_targets(calib_model, ft_test_loaders[task], task, device)

    # lock threshold from validation F1 sweep
    best_thr, val_best_f1 = sweep_threshold_f1(val_probs, val_y)
    calibrated_thresholds[task] = best_thr

    # eval on test using default vs locked threshold
    test_pred_05 = (test_probs >= 0.5).astype(int)
    test_pred_locked = (test_probs >= best_thr).astype(int)

    auroc_test = roc_auc_score(test_y, test_probs) if len(np.unique(test_y)) > 1 else 0.5

    row = {
        'task': task,
        'model_source': calib_source,
        'locked_threshold': round(best_thr, 3),
        'val_f1_at_locked': round(val_best_f1, 4),
        'test_auroc': round(float(auroc_test), 4),
        'test_f1_at_0.5': round(float(f1_score(test_y, test_pred_05, zero_division=0)), 4),
        'test_f1_at_locked': round(float(f1_score(test_y, test_pred_locked, zero_division=0)), 4),
        'test_acc_at_0.5': round(float(accuracy_score(test_y, test_pred_05)), 4),
        'test_acc_at_locked': round(float(accuracy_score(test_y, test_pred_locked)), 4),
        'delta_f1_locked_minus_0.5': round(
            float(f1_score(test_y, test_pred_locked, zero_division=0) - f1_score(test_y, test_pred_05, zero_division=0)), 4
        ),
    }
    threshold_rows.append(row)

threshold_calibration_summary = pd.DataFrame(threshold_rows)
print("\nLocked thresholds:")
print(calibrated_thresholds)
print("\nThreshold calibration summary (test impact):")
display(threshold_calibration_summary)

log_cell_end()


⏱  [2026-03-22 17:54:40] Starting: Cell 25 - Threshold Calibration
Calibration source model: Variant2

Locked thresholds:
{'herg': 0.33999999999999997, 'caco2': 0.24999999999999994}

Threshold calibration summary (test impact):


,task,model_source,locked_threshold,val_f1_at_locked,test_auroc,test_f1_at_0.5,test_f1_at_locked,test_acc_at_0.5,test_acc_at_locked,delta_f1_locked_minus_0.5
0,herg,Variant2,0.34,0.8519,0.7957,0.8394,0.8213,0.7634,0.7176,-0.0181
1,caco2,Variant2,0.25,0.7664,0.8716,0.8023,0.8040,0.8111,0.7833,0.0017


⏱  [2026-03-22 17:54:40] Completed in 0.1s


### 11.3 Constrained Threshold Calibration (F1 with Accuracy/Precision Guardrails)

This step selects validation thresholds that maximize F1 **while enforcing minimum accuracy and precision** to avoid degenerate low-threshold solutions.

In [8]:

# ============================================
# CONSTRAINED THRESHOLD CALIBRATION
# ============================================
log_cell_start("Cell 26 - Constrained Threshold Calibration")


def _precision_from_preds(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0


def sweep_threshold_constrained(probs, y_true, min_acc=0.5, min_precision=0.1, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)

    best = None
    for thr in grid:
        pred = (probs >= thr).astype(int)
        acc = accuracy_score(y_true, pred)
        prec = _precision_from_preds(y_true, pred)
        f1 = f1_score(y_true, pred, zero_division=0)

        if acc >= min_acc and prec >= min_precision:
            if (best is None) or (f1 > best['f1']):
                best = {'thr': float(thr), 'f1': float(f1), 'acc': float(acc), 'prec': float(prec), 'feasible': True}

    if best is None:
        # fallback (no feasible threshold under constraints)
        thr = 0.5
        pred = (probs >= thr).astype(int)
        best = {
            'thr': float(thr),
            'f1': float(f1_score(y_true, pred, zero_division=0)),
            'acc': float(accuracy_score(y_true, pred)),
            'prec': float(_precision_from_preds(y_true, pred)),
            'feasible': False,
        }
    return best


# Guardrails per task (can be tuned later)
threshold_constraints = {
    'herg': {'min_acc': 0.50, 'min_precision': 0.12},
    'caco2': {'min_acc': 0.50, 'min_precision': 0.50},
}

# Reuse the same calibration source model from previous step if available
if 'calib_model' not in globals():
    calib_model = ft2_model if 'ft2_model' in globals() else (ft_model if 'ft_model' in globals() else model)

constrained_thresholds = {}
constrained_rows = []

for task in FT_TASKS:
    val_probs, val_y = collect_probs_targets(calib_model, ft_val_loaders[task], task, device)
    test_probs, test_y = collect_probs_targets(calib_model, ft_test_loaders[task], task, device)

    cons = threshold_constraints[task]
    best_cons = sweep_threshold_constrained(
        val_probs,
        val_y,
        min_acc=cons['min_acc'],
        min_precision=cons['min_precision']
    )
    thr_cons = best_cons['thr']
    constrained_thresholds[task] = thr_cons

    thr_uncon = calibrated_thresholds[task] if 'calibrated_thresholds' in globals() and task in calibrated_thresholds else 0.5

    pred_05 = (test_probs >= 0.5).astype(int)
    pred_uncon = (test_probs >= thr_uncon).astype(int)
    pred_cons = (test_probs >= thr_cons).astype(int)

    constrained_rows.append({
        'task': task,
        'feasible_under_constraints': best_cons['feasible'],
        'thr_0.5': 0.5,
        'thr_unconstrained': round(float(thr_uncon), 3),
        'thr_constrained': round(float(thr_cons), 3),
        'test_auroc': round(float(roc_auc_score(test_y, test_probs) if len(np.unique(test_y)) > 1 else 0.5), 4),
        'f1_0.5': round(float(f1_score(test_y, pred_05, zero_division=0)), 4),
        'f1_unconstrained': round(float(f1_score(test_y, pred_uncon, zero_division=0)), 4),
        'f1_constrained': round(float(f1_score(test_y, pred_cons, zero_division=0)), 4),
        'acc_0.5': round(float(accuracy_score(test_y, pred_05)), 4),
        'acc_unconstrained': round(float(accuracy_score(test_y, pred_uncon)), 4),
        'acc_constrained': round(float(accuracy_score(test_y, pred_cons)), 4),
        'precision_constrained': round(float(_precision_from_preds(test_y, pred_cons)), 4),
    })

constrained_threshold_summary = pd.DataFrame(constrained_rows)

print("Constrained thresholds:")
print(constrained_thresholds)
print("\nConstrained calibration impact (test):")
display(constrained_threshold_summary)

log_cell_end()


⏱  [2026-03-22 17:54:45] Starting: Cell 26 - Constrained Threshold Calibration
Constrained thresholds:
{'herg': 0.33999999999999997, 'caco2': 0.24999999999999994}

Constrained calibration impact (test):


,task,feasible_under_constraints,thr_0.5,thr_unconstrained,thr_constrained,test_auroc,f1_0.5,f1_unconstrained,f1_constrained,acc_0.5,acc_unconstrained,acc_constrained,precision_constrained
0,herg,True,0.5,0.34,0.34,0.7957,0.8394,0.8213,0.8213,0.7634,0.7176,0.7176,0.7143
1,caco2,True,0.5,0.25,0.25,0.8716,0.8023,0.8040,0.8040,0.8111,0.7833,0.7833,0.7273


⏱  [2026-03-22 17:54:45] Completed in 0.1s


### 11.4 Production Threshold Policy (Automatic Selection)

Decision rule per task:
1. Prefer constrained threshold if it improves F1 over 0.5 and keeps accuracy above minimum guardrail.
2. Otherwise use 0.5 baseline.

This avoids selecting unstable low-threshold policies that inflate one metric while collapsing operational behavior.

In [9]:

# ============================================
# AUTO-SELECT PRODUCTION THRESHOLD POLICY
# ============================================
log_cell_start("Cell 27 - Threshold Policy Decision")

if 'constrained_threshold_summary' not in globals():
    raise ValueError("Run constrained threshold calibration cell first.")

# Task-specific operational guardrails
policy_min_acc = {
    'herg': 0.45,
    'caco2': 0.50,
}

policy_rows = []
production_thresholds = {}

for _, r in constrained_threshold_summary.iterrows():
    task = r['task']

    # Candidate A: baseline 0.5
    f1_base = float(r['f1_0.5'])
    acc_base = float(r['acc_0.5'])

    # Candidate B: constrained calibrated threshold
    f1_cons = float(r['f1_constrained'])
    acc_cons = float(r['acc_constrained'])
    thr_cons = float(r['thr_constrained'])

    min_acc_req = float(policy_min_acc.get(task, 0.5))

    # Decision logic
    use_constrained = (acc_cons >= min_acc_req) and (f1_cons > f1_base)

    if use_constrained:
        selected_policy = 'constrained'
        selected_thr = thr_cons
        selected_f1 = f1_cons
        selected_acc = acc_cons
    else:
        selected_policy = 'baseline_0.5'
        selected_thr = 0.5
        selected_f1 = f1_base
        selected_acc = acc_base

    production_thresholds[task] = float(selected_thr)

    policy_rows.append({
        'task': task,
        'selected_policy': selected_policy,
        'selected_threshold': round(float(selected_thr), 3),
        'selected_f1': round(float(selected_f1), 4),
        'selected_acc': round(float(selected_acc), 4),
        'baseline_f1': round(f1_base, 4),
        'constrained_f1': round(f1_cons, 4),
        'baseline_acc': round(acc_base, 4),
        'constrained_acc': round(acc_cons, 4),
        'min_acc_required': round(min_acc_req, 3),
    })

threshold_policy_decision = pd.DataFrame(policy_rows)

print("✅ Production threshold policy selected")
print("Locked production thresholds:")
print(production_thresholds)
print("\nDecision table:")
display(threshold_policy_decision)

log_cell_end()


⏱  [2026-03-22 17:54:49] Starting: Cell 27 - Threshold Policy Decision
✅ Production threshold policy selected
Locked production thresholds:
{'herg': 0.5, 'caco2': 0.25}

Decision table:


,task,selected_policy,selected_threshold,selected_f1,selected_acc,baseline_f1,constrained_f1,baseline_acc,constrained_acc,min_acc_required
0,herg,baseline_0.5,0.50,0.8394,0.7634,0.8394,0.8213,0.7634,0.7176,0.45
1,caco2,constrained,0.25,0.8040,0.7833,0.8023,0.8040,0.8111,0.7833,0.50


⏱  [2026-03-22 17:54:49] Completed in 0.0s


### 11.5 Final Report Metrics (Locked Production Thresholds)

This cell computes report-ready **test metrics** using the locked `production_thresholds` only.

- No threshold sweep
- No additional weight updates
- Uses the selected production model (`calib_model` if available, otherwise `model`)

In [10]:
# ============================================
# FINAL REPORT METRICS USING PRODUCTION THRESHOLDS
# ============================================
log_cell_start("Cell 28 - Final Report Metrics (Locked Thresholds)")

if 'production_thresholds' not in globals():
    raise ValueError("Run the Production Threshold Policy cell first to create production_thresholds.")

# Pick production model source (same hierarchy as calibration workflow)
if 'calib_model' in globals():
    production_model = calib_model
    production_model_source = 'calib_model'
else:
    production_model = model
    production_model_source = 'model'

print(f"Using production model source: {production_model_source}")
print(f"Locked thresholds: {production_thresholds}")

# Classification report rows (locked thresholds only)
classification_rows = []
for task in FT_TASKS:
    probs_test, y_test = collect_probs_targets(production_model, ft_test_loaders[task], task, device)
    thr = float(production_thresholds.get(task, 0.5))
    pred = (probs_test >= thr).astype(int)

    auroc = roc_auc_score(y_test, probs_test) if len(np.unique(y_test)) > 1 else 0.5
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)
    precision = _precision_from_preds(y_test, pred)
    n = len(y_test)
    pos_rate = float(np.mean(y_test))

    classification_rows.append({
        'task': task,
        'threshold_locked': round(thr, 3),
        'test_samples': int(n),
        'test_pos_rate': round(pos_rate, 4),
        'test_auroc': round(float(auroc), 4),
        'test_accuracy': round(float(acc), 4),
        'test_precision': round(float(precision), 4),
        'test_f1': round(float(f1), 4),
    })

classification_report_locked = pd.DataFrame(classification_rows)

# Regression metrics from the same production model for completeness
criterion_report = MultiTaskLoss(config)
reg_results = validate(
    production_model,
    {k: v for k, v in test_loaders.items() if k in ['binding', 'clearance']},
    criterion_report,
    device
)

regression_report = pd.DataFrame([
    {
        'task': task,
        'test_rmse': round(float(reg_results[task]['rmse']), 4),
        'test_r2': round(float(reg_results[task]['r2']), 4),
        'test_mae': round(float(reg_results[task]['mae']), 4),
    }
    for task in ['binding', 'clearance']
])

print("\n✅ Final report metrics (classification; locked thresholds)")
display(classification_report_locked)

print("\n✅ Final report metrics (regression)")
display(regression_report)

# Optional single combined view for docs export
combined_report_rows = []
for _, r in classification_report_locked.iterrows():
    combined_report_rows.append({
        'task': r['task'],
        'type': 'classification',
        'threshold': r['threshold_locked'],
        'auroc': r['test_auroc'],
        'accuracy': r['test_accuracy'],
        'precision': r['test_precision'],
        'f1': r['test_f1'],
        'rmse': np.nan,
        'r2': np.nan,
        'mae': np.nan,
    })

for _, r in regression_report.iterrows():
    combined_report_rows.append({
        'task': r['task'],
        'type': 'regression',
        'threshold': np.nan,
        'auroc': np.nan,
        'accuracy': np.nan,
        'precision': np.nan,
        'f1': np.nan,
        'rmse': r['test_rmse'],
        'r2': r['test_r2'],
        'mae': r['test_mae'],
    })

final_report_metrics = pd.DataFrame(combined_report_rows)
print("\n📄 Unified final metrics table:")
display(final_report_metrics)

log_cell_end()


⏱  [2026-03-22 17:54:53] Starting: Cell 28 - Final Report Metrics (Locked Thresholds)
Using production model source: calib_model
Locked thresholds: {'herg': 0.5, 'caco2': 0.25}

✅ Final report metrics (classification; locked thresholds)


,task,threshold_locked,test_samples,test_pos_rate,test_auroc,test_accuracy,test_precision,test_f1
0,herg,0.50,131,0.6718,0.7957,0.7634,0.7714,0.8394
1,caco2,0.25,180,0.4944,0.8716,0.7833,0.7273,0.8040



✅ Final report metrics (regression)


,task,test_rmse,test_r2,test_mae
0,binding,0.7959,0.3684,0.6161
1,clearance,0.9160,0.1143,0.7047



📄 Unified final metrics table:


,task,type,threshold,auroc,accuracy,precision,f1,rmse,r2,mae
0,herg,classification,0.50,0.7957,0.7634,0.7714,0.8394,NaN,NaN,NaN
1,caco2,classification,0.25,0.8716,0.7833,0.7273,0.8040,NaN,NaN,NaN
2,binding,regression,NaN,NaN,NaN,NaN,NaN,0.7959,0.3684,0.6161
3,clearance,regression,NaN,NaN,NaN,NaN,NaN,0.9160,0.1143,0.7047


⏱  [2026-03-22 17:54:53] Completed in 0.1s


---
## 12. Split-Leakage Mitigation

The data-quality gate (Section 4.5) detected exact-feature-row overlap across train/val/test splits, primarily in the binding task.

This section fixes the root cause by **deduplicating on feature rows before splitting**, then rerunning the full workflow:

1. Dedup all task arrays on feature hash
2. Re-run quality gate → confirm zero leakage
3. Retrain model on clean splits (same config)
4. Apply locked production thresholds and compare against March 8 baseline

In [11]:
# ============================================
# STEP 1: DEDUPLICATION BEFORE SPLIT
# ============================================
log_cell_start("Cell 29 - Dedup Before Split")

import hashlib


def row_hash_index(X):
    """Return index array that retains only the first occurrence of each unique feature row."""
    X = np.ascontiguousarray(X.astype(np.float32))
    seen = {}
    keep = []
    for i, row in enumerate(X):
        h = hashlib.md5(row.tobytes()).hexdigest()
        if h not in seen:
            seen[h] = i
            keep.append(i)
    return np.array(keep, dtype=np.int64)


dedup_stats = []
tasks_raw_clean = {}

for task, (X_t, y_t) in tasks_raw.items():
    X_t = np.asarray(X_t, dtype=np.float32)
    y_t = np.asarray(y_t, dtype=np.float32)
    n_before = len(X_t)
    keep_idx = row_hash_index(X_t)
    X_clean = X_t[keep_idx]
    y_clean = y_t[keep_idx]
    n_after = len(X_clean)
    removed = n_before - n_after
    tasks_raw_clean[task] = (X_clean, y_clean)
    dedup_stats.append({
        'task': task,
        'n_before': n_before,
        'n_after': n_after,
        'duplicates_removed': removed,
        'pct_removed': round(100.0 * removed / n_before, 2),
    })

dedup_summary = pd.DataFrame(dedup_stats)
print("✅ Deduplication complete")
display(dedup_summary)

log_cell_end()


⏱  [2026-03-22 17:54:58] Starting: Cell 29 - Dedup Before Split
✅ Deduplication complete


,task,n_before,n_after,duplicates_removed,pct_removed
0,binding,3410,3282,128,3.75
1,herg,655,606,49,7.48
2,caco2,900,814,86,9.56
3,clearance,1213,1000,213,17.56


⏱  [2026-03-22 17:54:58] Completed in 0.1s


In [12]:
# ============================================
# STEP 2: QUALITY GATE RE-RUN ON CLEAN SPLITS
# ============================================
log_cell_start("Cell 30 - Quality Gate (Clean Splits)")

clean_quality_rows = []

for task, (X_task, y_task) in tasks_raw_clean.items():
    X_task = np.asarray(X_task, dtype=np.float32)
    y_task = np.asarray(y_task, dtype=np.float32)

    n_samples, n_features = X_task.shape
    n_nan_X = int(np.isnan(X_task).sum())
    n_inf_X = int(np.isinf(X_task).sum())
    n_nan_y = int(np.isnan(y_task).sum())
    n_inf_y = int(np.isinf(y_task).sum())

    unique_rows = np.unique(X_task, axis=0).shape[0]
    dup_rows = int(n_samples - unique_rows)

    X_tr, X_va, X_te, y_tr, y_va, y_te = _split_like_training(X_task, y_task, config, random_state=42)

    h_tr = np.unique(_row_hash(X_tr))
    h_va = np.unique(_row_hash(X_va))
    h_te = np.unique(_row_hash(X_te))
    leak_tr_va = int(np.intersect1d(h_tr, h_va).size)
    leak_tr_te = int(np.intersect1d(h_tr, h_te).size)
    leak_va_te = int(np.intersect1d(h_va, h_te).size)

    if task_types[task]:
        tr_mu, va_mu, te_mu = float(np.mean(y_tr)), float(np.mean(y_va)), float(np.mean(y_te))
        tr_sd = float(np.std(y_tr) + 1e-8)
        clean_quality_rows.append({
            'task': task, 'type': 'regression', 'samples': n_samples,
            'nan_inf_total': n_nan_X + n_inf_X + n_nan_y + n_inf_y,
            'dup_rows': dup_rows,
            'leak_tr_val': leak_tr_va, 'leak_tr_test': leak_tr_te, 'leak_val_test': leak_va_te,
            'val_drift_sigma': round(abs(va_mu - tr_mu) / tr_sd, 3),
            'test_drift_sigma': round(abs(te_mu - tr_mu) / tr_sd, 3),
        })
    else:
        clean_quality_rows.append({
            'task': task, 'type': 'classification', 'samples': n_samples,
            'nan_inf_total': n_nan_X + n_inf_X + n_nan_y + n_inf_y,
            'dup_rows': dup_rows,
            'leak_tr_val': leak_tr_va, 'leak_tr_test': leak_tr_te, 'leak_val_test': leak_va_te,
            'val_pos_delta': round(abs(float(np.mean(y_va)) - float(np.mean(y_tr))), 4),
            'test_pos_delta': round(abs(float(np.mean(y_te)) - float(np.mean(y_tr))), 4),
        })

clean_quality_df = pd.DataFrame(clean_quality_rows)
print("✅ Quality gate on clean splits:")
display(clean_quality_df)

# Gate results
nan_ok = bool((clean_quality_df['nan_inf_total'] == 0).all())
leak_ok = bool(((clean_quality_df['leak_tr_val'] == 0) &
                (clean_quality_df['leak_tr_test'] == 0) &
                (clean_quality_df['leak_val_test'] == 0)).all())
print(f"\n  No NaN/Inf:      {'PASS' if nan_ok else 'FAIL'}")
print(f"  No split leakage: {'PASS' if leak_ok else 'FAIL'}")
print(f"\n🎯 CLEAN QUALITY GATE: {'PASS' if nan_ok and leak_ok else 'FAIL'}")

log_cell_end()


⏱  [2026-03-22 17:55:02] Starting: Cell 30 - Quality Gate (Clean Splits)
✅ Quality gate on clean splits:


,task,type,samples,nan_inf_total,dup_rows,leak_tr_val,leak_tr_test,leak_val_test,val_drift_sigma,test_drift_sigma,val_pos_delta,test_pos_delta
0,binding,regression,3282,0,0,0,0,0,0.098,0.018,NaN,NaN
1,herg,classification,606,0,0,0,0,0,NaN,NaN,0.0124,0.0122
2,caco2,classification,814,0,0,0,0,0,NaN,NaN,0.0234,0.0328
3,clearance,regression,1000,0,0,0,0,0,0.012,0.017,NaN,NaN



  No NaN/Inf:      PASS
  No split leakage: PASS

🎯 CLEAN QUALITY GATE: PASS
⏱  [2026-03-22 17:55:02] Completed in 0.3s


In [13]:
# ============================================
# STEP 3: REBUILD DATALOADERS FROM CLEAN SPLITS
# ============================================
log_cell_start("Cell 31 - Clean DataLoaders")

# Recompute class weights from cleaned arrays
clean_y_herg  = tasks_raw_clean['herg'][1]
clean_y_caco2 = tasks_raw_clean['caco2'][1]

config_clean = dict(config)  # copy config so we don't mutate the original
config_clean['herg_pos_weight']  = compute_pos_weight(clean_y_herg)
config_clean['caco2_pos_weight'] = compute_pos_weight(clean_y_caco2)

train_loaders_clean, val_loaders_clean, test_loaders_clean = {}, {}, {}
feat_scalers_clean, target_scalers_clean = {}, {}

for task, (X_c, y_c) in tasks_raw_clean.items():
    tr, va, te, fs, ts = make_loaders(X_c, y_c, config_clean, is_regression=task_types[task])
    train_loaders_clean[task]  = tr
    val_loaders_clean[task]    = va
    test_loaders_clean[task]   = te
    feat_scalers_clean[task]   = fs
    target_scalers_clean[task] = ts

print("✅ Clean DataLoaders built:")
for task in tasks_raw_clean:
    n_tr = len(train_loaders_clean[task].dataset)
    n_va = len(val_loaders_clean[task].dataset)
    n_te = len(test_loaders_clean[task].dataset)
    print(f"  {task:<12}  train={n_tr}, val={n_va}, test={n_te}")
print(f"\n  herg_pos_weight:  {config_clean['herg_pos_weight']:.3f}")
print(f"  caco2_pos_weight: {config_clean['caco2_pos_weight']:.3f}")

log_cell_end()


⏱  [2026-03-22 17:55:06] Starting: Cell 31 - Clean DataLoaders
✅ Clean DataLoaders built:
  binding       train=2296, val=329, test=657
  herg          train=423, val=61, test=122
  caco2         train=569, val=82, test=163
  clearance     train=700, val=100, test=200

  herg_pos_weight:  1.000
  caco2_pos_weight: 1.000
⏱  [2026-03-22 17:55:06] Completed in 0.1s


In [14]:
# ============================================
# STEP 4: RETRAIN ON CLEAN SPLITS
# ============================================
log_cell_start("Cell 32 - Retrain (Clean Splits)")

import time

model_clean = MultiTaskPKPDModel(config_clean).to(device)

t0_clean = time.time()
model_clean, history_clean = train_model(
    model_clean,
    train_loaders_clean,
    val_loaders_clean,
    config_clean,
    device,
)
print(f"\nRetrain time: {time.time() - t0_clean:.1f}s")
log_elapsed("Retrain on clean splits complete")

log_cell_end()


⏱  [2026-03-22 17:55:11] Starting: Cell 32 - Retrain (Clean Splits)
Training for up to 300 epochs (patience=40)...
Epoch 1/300
  Train Loss: 3.1651  Val Loss: 3.0589
    binding   : R²=0.028, RMSE=0.940
    herg      : AUROC=0.680, Acc=0.689
    caco2     : AUROC=0.908, Acc=0.744
    clearance : R²=0.031, RMSE=0.996
Epoch 10/300
  Train Loss: 0.7206  Val Loss: 2.4241
    binding   : R²=0.413, RMSE=0.731
    herg      : AUROC=0.816, Acc=0.770
    caco2     : AUROC=0.865, Acc=0.744
    clearance : R²=0.128, RMSE=0.944
Epoch 20/300
  Train Loss: 0.4039  Val Loss: 2.4152
    binding   : R²=0.484, RMSE=0.685
    herg      : AUROC=0.776, Acc=0.738
    caco2     : AUROC=0.894, Acc=0.805
    clearance : R²=0.259, RMSE=0.871
Epoch 30/300
  Train Loss: 0.2667  Val Loss: 2.4884
    binding   : R²=0.469, RMSE=0.695
    herg      : AUROC=0.769, Acc=0.721
    caco2     : AUROC=0.885, Acc=0.829
    clearance : R²=0.266, RMSE=0.867
Epoch 40/300
  Train Loss: 0.2362  Val Loss: 2.5437
    binding   : R²

In [15]:
# ============================================
# STEP 5: LOCKED-THRESHOLD REPORT & COMPARISON vs MARCH 8 BASELINE
# ============================================
log_cell_start("Cell 33 - Clean Splits Locked-Threshold Report")

# ── Classification metrics (locked production thresholds) ─────────────────
criterion_clean = MultiTaskLoss(config_clean)

# Collect test probabilities from new model
clean_cls_rows = []
for task in FT_TASKS:
    probs_c, y_c = collect_probs_targets(model_clean, test_loaders_clean[task], task, device)
    thr = float(production_thresholds.get(task, 0.5))  # reuse locked thresholds
    pred_c = (probs_c >= thr).astype(int)

    auroc_c = roc_auc_score(y_c, probs_c) if len(np.unique(y_c)) > 1 else 0.5
    acc_c   = accuracy_score(y_c, pred_c)
    f1_c    = f1_score(y_c, pred_c, zero_division=0)
    prec_c  = _precision_from_preds(y_c, pred_c)

    clean_cls_rows.append({
        'task': task, 'threshold': thr,
        'auroc': round(float(auroc_c), 4),
        'accuracy': round(float(acc_c), 4),
        'precision': round(float(prec_c), 4),
        'f1': round(float(f1_c), 4),
    })

clean_cls_df = pd.DataFrame(clean_cls_rows)

# ── Regression metrics ────────────────────────────────────────────────────
clean_reg_results = validate(
    model_clean,
    {k: v for k, v in test_loaders_clean.items() if k in ['binding', 'clearance']},
    criterion_clean,
    device,
)
clean_reg_rows = [
    {'task': t,
     'rmse': round(float(clean_reg_results[t]['rmse']), 4),
     'r2':   round(float(clean_reg_results[t]['r2']), 4),
     'mae':  round(float(clean_reg_results[t]['mae']), 4)}
    for t in ['binding', 'clearance']
]
clean_reg_df = pd.DataFrame(clean_reg_rows)

print("✅ Clean-split locked-threshold results (classification):")
display(clean_cls_df)
print("\n✅ Clean-split results (regression):")
display(clean_reg_df)

# ── Delta vs March 8 baseline ─────────────────────────────────────────────
MAR8 = {
    'herg':      {'auroc': 0.4836, 'f1': 0.1962, 'accuracy': 0.4931},
    'caco2':     {'auroc': 0.4719, 'f1': 0.7183, 'accuracy': 0.5604},
    'binding':   {'r2': 0.0019,  'rmse': 1.0069},
    'clearance': {'r2': -0.0129, 'rmse': 0.9766},
}

print("\n📊 Delta vs March 8 locked-threshold baseline:")
delta_rows = []
for r in clean_cls_rows:
    t = r['task']
    delta_rows.append({
        'task': t, 'type': 'classification',
        'auroc_mar8': MAR8[t]['auroc'], 'auroc_clean': r['auroc'],
        'Δauroc': round(r['auroc'] - MAR8[t]['auroc'], 4),
        'f1_mar8': MAR8[t]['f1'],       'f1_clean': r['f1'],
        'Δf1':    round(r['f1'] - MAR8[t]['f1'], 4),
    })
for r in clean_reg_rows:
    t = r['task']
    delta_rows.append({
        'task': t, 'type': 'regression',
        'r2_mar8': MAR8[t]['r2'],   'r2_clean': r['r2'],
        'Δr2':    round(r['r2'] - MAR8[t]['r2'], 4),
        'rmse_mar8': MAR8[t]['rmse'], 'rmse_clean': r['rmse'],
        'Δrmse':  round(r['rmse'] - MAR8[t]['rmse'], 4),
    })

delta_df = pd.DataFrame(delta_rows)
display(delta_df)
print("\n(Positive Δauroc / Δr2 = improvement; negative Δrmse = improvement)")

log_cell_end()


⏱  [2026-03-22 17:55:17] Starting: Cell 33 - Clean Splits Locked-Threshold Report
✅ Clean-split locked-threshold results (classification):


,task,threshold,auroc,accuracy,precision,f1
0,herg,0.50,0.7332,0.7295,0.7553,0.8114
1,caco2,0.25,0.8650,0.7485,0.6935,0.8075



✅ Clean-split results (regression):


,task,rmse,r2,mae
0,binding,0.7177,0.4801,0.5491
1,clearance,0.8985,0.1517,0.6415



📊 Delta vs March 8 locked-threshold baseline:


,task,type,auroc_mar8,auroc_clean,Δauroc,f1_mar8,f1_clean,Δf1,r2_mar8,r2_clean,Δr2,rmse_mar8,rmse_clean,Δrmse
0,herg,classification,0.4836,0.7332,0.2496,0.1962,0.8114,0.6152,NaN,NaN,NaN,NaN,NaN,NaN
1,caco2,classification,0.4719,0.8650,0.3931,0.7183,0.8075,0.0892,NaN,NaN,NaN,NaN,NaN,NaN
2,binding,regression,NaN,NaN,NaN,NaN,NaN,NaN,0.0019,0.4801,0.4782,1.0069,0.7177,-0.2892
3,clearance,regression,NaN,NaN,NaN,NaN,NaN,NaN,-0.0129,0.1517,0.1646,0.9766,0.8985,-0.0781



(Positive Δauroc / Δr2 = improvement; negative Δrmse = improvement)
⏱  [2026-03-22 17:55:17] Completed in 0.0s


---
## Save Phase 3C Artifacts

Save the production model, thresholds, and clean-split state for `phase3d_experiments.ipynb`.


In [16]:
# ============================================
# SAVE PHASE 3C ARTIFACTS FOR PHASE 3D
# ============================================
import pickle, os, json as _json

ARTIFACT_DIR_3C = 'data/processed/phase3c_artifacts'
os.makedirs(ARTIFACT_DIR_3C, exist_ok=True)

# 1. Save production model (best after fine-tuning / calibration)
prod_model = calib_model if 'calib_model' in dir() else model
torch.save(prod_model.state_dict(), os.path.join(ARTIFACT_DIR_3C, 'production_model_state.pt'))

# 2. Save production thresholds
with open(os.path.join(ARTIFACT_DIR_3C, 'production_thresholds.json'), 'w') as f:
    _json.dump({k: float(v) for k, v in production_thresholds.items()}, f, indent=2)

# 3. Save config (may have been updated)
with open(os.path.join(ARTIFACT_DIR_3C, 'config.json'), 'w') as f:
    _json.dump(config, f, indent=2)

# 4. Save loader state + feature meta
with open(os.path.join(ARTIFACT_DIR_3C, 'loader_state.pkl'), 'wb') as f:
    pickle.dump({
        'tasks_raw': tasks_raw, 'task_types': task_types,
        'feat_scalers': feat_scalers, 'target_scalers': target_scalers,
    }, f)

with open(os.path.join(ARTIFACT_DIR_3C, 'feature_meta.pkl'), 'wb') as f:
    pickle.dump({
        'FEATURE_COLS': FEATURE_COLS, 'PHYSICO_FEATURES': PHYSICO_FEATURES,
        'FINGERPRINT_COLS': FINGERPRINT_COLS, 'N_BITS': N_BITS, 'FEAT_DIM': FEAT_DIM,
    }, f)

# 5. Save raw arrays
with open(os.path.join(ARTIFACT_DIR_3C, 'raw_arrays.pkl'), 'wb') as f:
    pickle.dump({
        'X_binding': X_binding, 'y_binding': y_binding,
        'X_herg': X_herg, 'y_herg': y_herg,
        'X_caco2': X_caco2, 'y_caco2': y_caco2,
        'X_clearance': X_clearance, 'y_clearance': y_clearance,
    }, f)

# 6. Save model info
with open(os.path.join(ARTIFACT_DIR_3C, 'model_info.pkl'), 'wb') as f:
    pickle.dump({'total_params': total_params, 'trainable_params': trainable_params}, f)

# 7. Save final report metrics if available
if 'final_report_metrics' in dir():
    final_report_metrics.to_csv(os.path.join(ARTIFACT_DIR_3C, 'final_report_metrics.csv'), index=False)

print(f'\n\u2705 Phase 3C artifacts saved to {ARTIFACT_DIR_3C}/')
for fname in sorted(os.listdir(ARTIFACT_DIR_3C)):
    fsize = os.path.getsize(os.path.join(ARTIFACT_DIR_3C, fname))
    print(f'  {fname:40s} {fsize/1024:.1f} KB')
print('\n\u27a1\ufe0f  Continue with phase3d_experiments.ipynb')



✅ Phase 3C artifacts saved to data/processed/phase3c_artifacts/
  config.json                              0.5 KB
  feature_meta.pkl                         29.4 KB
  final_report_metrics.csv                 0.3 KB
  loader_state.pkl                         49714.4 KB
  model_info.pkl                           19.0 KB
  production_model_state.pt                1194.5 KB
  production_thresholds.json               0.0 KB
  raw_arrays.pkl                           49521.0 KB

➡️  Continue with phase3d_experiments.ipynb


---
## Phase 3C Execution Timeline Summary


In [17]:

# ============================================
# PHASE 3C — EXECUTION TIMELINE SUMMARY
# ============================================
import datetime

print("=" * 88)
print("  PHASE 3C — CELL EXECUTION TIMELINE")
print("=" * 88)

if not CELL_EXEC_LOG:
    print("  (no cells logged yet)")
else:
    print(f"  {'#':<4} {'Section':<45} {'Start':>8}  {'End':>8}  {'Dur (s)':>8}")
    print("-" * 88)
    total_compute = 0.0
    for i, entry in enumerate(CELL_EXEC_LOG, 1):
        start_str = entry['start'].strftime('%H:%M:%S') if entry['start'] else '—'
        end_str   = entry['end'].strftime('%H:%M:%S') if entry['end'] else 'running…'
        dur_str   = f"{entry['duration_s']:.1f}" if entry['duration_s'] is not None else '—'
        if entry['duration_s'] is not None:
            total_compute += entry['duration_s']
        section = entry['section'][:44]
        print(f"  {i:<4} {section:<45} {start_str:>8}  {end_str:>8}  {dur_str:>8}")
    print("-" * 88)

    # Totals
    wall_clock = (datetime.datetime.now() - SESSION_START).total_seconds()
    h_c, rem_c = divmod(int(total_compute), 3600)
    m_c, s_c   = divmod(rem_c, 60)
    h_w, rem_w = divmod(int(wall_clock), 3600)
    m_w, s_w   = divmod(rem_w, 60)

    print(f"  {'Total compute time:':<55} {h_c}h {m_c}m {s_c}s  ({total_compute:.1f}s)")
    print(f"  {'Wall-clock time (session start → now):':<55} {h_w}h {m_w}m {s_w}s  ({wall_clock:.1f}s)")
    print(f"  {'Session started:':<55} {SESSION_START.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  {'Summary generated:':<55} {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  {'Cells logged:':<55} {len(CELL_EXEC_LOG)}")
print("=" * 88)


  PHASE 3C — CELL EXECUTION TIMELINE
  #    Section                                          Start       End   Dur (s)
----------------------------------------------------------------------------------------
  1    Load Phase 3B Artifacts                       17:53:59  17:53:59       0.3
  2    Rebuild Model & DataLoaders                   17:54:07  17:54:07       0.2
  3    Rebuild Core Functions                        17:54:14  17:54:14       0.0
  4    Cell 22 - FineTune Setup                      17:54:18  17:54:18       0.0
  5    Cell 23 - FineTune Run                        17:54:22  17:54:23       1.2
  6    Cell 24 - FineTune Variant2                   17:54:35  17:54:36       1.1
  7    Cell 25 - Threshold Calibration               17:54:40  17:54:40       0.1
  8    Cell 26 - Constrained Threshold Calibration   17:54:45  17:54:45       0.1
  9    Cell 27 - Threshold Policy Decision           17:54:49  17:54:49       0.0
  10   Cell 28 - Final Report Metrics (Locked Thres  1